# SQL-based Data Analysis using Filtering, Aggregation, and Basic Business Queries

**Name:** Ayush Rai  
**Week:** 2  
**Topic:** SQL Analysis — ShopEase E-commerce Database  
**Tool:** Python (sqlite3 + pandas)

---

### Objective
Analyze sales data using SQL with filtering, aggregation, and business queries on a simulated e-commerce database containing customers, products, orders, and order items.


## Setup — Import Libraries and Connect to Database

In [1]:
import sqlite3
import pandas as pd

# connect to in-memory SQLite database
conn = sqlite3.connect(":memory:")
cur = conn.cursor()

def show(query):
    return pd.read_sql_query(query, conn)

print("Database connected.")


Database connected.


## Create Tables

In [2]:
cur.executescript("""
CREATE TABLE customers (
    customer_id  INTEGER PRIMARY KEY,
    first_name   TEXT NOT NULL,
    last_name    TEXT NOT NULL,
    email        TEXT UNIQUE NOT NULL,
    city         TEXT NOT NULL,
    state        TEXT NOT NULL,
    join_date    TEXT NOT NULL,
    is_premium   INTEGER DEFAULT 0
);

CREATE TABLE products (
    product_id   INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    category     TEXT NOT NULL,
    brand        TEXT NOT NULL,
    unit_price   REAL NOT NULL CHECK (unit_price > 0),
    stock_qty    INTEGER NOT NULL DEFAULT 0 CHECK (stock_qty >= 0)
);

CREATE TABLE orders (
    order_id     INTEGER PRIMARY KEY,
    customer_id  INTEGER NOT NULL,
    order_date   TEXT NOT NULL,
    status       TEXT NOT NULL DEFAULT 'Pending'
                 CHECK (status IN ('Pending','Shipped','Delivered','Cancelled')),
    total_amount REAL NOT NULL CHECK (total_amount >= 0),
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

CREATE TABLE order_items (
    item_id      INTEGER PRIMARY KEY,
    order_id     INTEGER NOT NULL,
    product_id   INTEGER NOT NULL,
    quantity     INTEGER NOT NULL CHECK (quantity > 0),
    unit_price   REAL NOT NULL CHECK (unit_price > 0),
    discount_pct REAL DEFAULT 0 CHECK (discount_pct BETWEEN 0 AND 100),
    FOREIGN KEY (order_id)   REFERENCES orders(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
""")
print("All tables created successfully.")


All tables created successfully.


## Load Sample Data

In [3]:
cur.executescript("""
INSERT INTO customers VALUES
(101,'Aarav','Sharma','aarav.s@email.com','Mumbai','Maharashtra','2024-01-15',1),
(102,'Priya','Patel','priya.p@email.com','Ahmedabad','Gujarat','2024-02-20',0),
(103,'Rohan','Gupta','rohan.g@email.com','Delhi','Delhi','2024-03-10',1),
(104,'Sneha','Reddy','sneha.r@email.com','Hyderabad','Telangana','2024-04-05',0),
(105,'Vikram','Singh','vikram.s@email.com','Jaipur','Rajasthan','2024-05-12',1),
(106,'Ananya','Iyer','ananya.i@email.com','Chennai','Tamil Nadu','2024-06-18',0),
(107,'Karan','Mehta','karan.m@email.com','Pune','Maharashtra','2024-07-22',1),
(108,'Divya','Nair','divya.n@email.com','Kochi','Kerala','2024-08-30',0);

INSERT INTO products VALUES
(201,'Wireless Earbuds','Electronics','BoAt',1499.00,250),
(202,'Cotton T-Shirt','Clothing','Levis',799.00,500),
(203,'Smart Watch','Electronics','Noise',2999.00,150),
(204,'Running Shoes','Clothing','Nike',4599.00,120),
(205,'Bluetooth Speaker','Electronics','JBL',3499.00,200),
(206,'Bedsheet Set','Home','Spaces',1299.00,300),
(207,'Laptop Stand','Electronics','AmazonBasics',899.00,180),
(208,'Cushion Covers (Set)','Home','HomeCenter',599.00,400);

INSERT INTO orders VALUES
(1001,101,'2024-08-01','Delivered',4498.00),
(1002,102,'2024-08-03','Delivered',799.00),
(1003,103,'2024-08-05','Shipped',7498.00),
(1004,101,'2024-08-10','Delivered',3499.00),
(1005,104,'2024-08-12','Cancelled',2999.00),
(1006,105,'2024-08-15','Delivered',5898.00),
(1007,106,'2024-08-18','Pending',1299.00),
(1008,103,'2024-08-20','Delivered',899.00),
(1009,107,'2024-08-25','Shipped',6098.00),
(1010,108,'2024-08-28','Delivered',1598.00);

INSERT INTO order_items VALUES
(5001,1001,201,2,1499.00,0),
(5002,1001,207,1,899.00,10),
(5003,1002,202,1,799.00,0),
(5004,1003,203,1,2999.00,0),
(5005,1003,204,1,4599.00,5),
(5006,1004,205,1,3499.00,0),
(5007,1005,203,1,2999.00,0),
(5008,1006,201,1,1499.00,10),
(5009,1006,204,1,4599.00,5),
(5010,1007,206,1,1299.00,0),
(5011,1008,207,1,899.00,0),
(5012,1009,205,1,3499.00,0),
(5013,1009,208,2,599.00,15),
(5014,1010,206,1,1299.00,0),
(5015,1010,208,1,599.00,0);
""")
print("Sample data loaded successfully.")


Sample data loaded successfully.


## Quick Data Overview

In [4]:
# checking row counts in each table
print("customers :", cur.execute("SELECT COUNT(*) FROM customers").fetchone()[0])
print("products  :", cur.execute("SELECT COUNT(*) FROM products").fetchone()[0])
print("orders    :", cur.execute("SELECT COUNT(*) FROM orders").fetchone()[0])
print("order_items:", cur.execute("SELECT COUNT(*) FROM order_items").fetchone()[0])


customers : 8
products  : 8
orders    : 10
order_items: 15


---
## Section A — SQL Basics

### Q1. Display all columns and rows from the customers table.

In [5]:
show("SELECT * FROM customers")

,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
1,102,Priya,Patel,priya.p@email.com,Ahmedabad,Gujarat,2024-02-20,0
2,103,Rohan,Gupta,rohan.g@email.com,Delhi,Delhi,2024-03-10,1
3,104,Sneha,Reddy,sneha.r@email.com,Hyderabad,Telangana,2024-04-05,0
4,105,Vikram,Singh,vikram.s@email.com,Jaipur,Rajasthan,2024-05-12,1
5,106,Ananya,Iyer,ananya.i@email.com,Chennai,Tamil Nadu,2024-06-18,0
6,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1
7,108,Divya,Nair,divya.n@email.com,Kochi,Kerala,2024-08-30,0


### Q2. Retrieve only first_name, last_name, and city of all customers.

In [6]:
show("SELECT first_name, last_name, city FROM customers")

,first_name,last_name,city
0,Aarav,Sharma,Mumbai
1,Priya,Patel,Ahmedabad
2,Rohan,Gupta,Delhi
3,Sneha,Reddy,Hyderabad
4,Vikram,Singh,Jaipur
5,Ananya,Iyer,Chennai
6,Karan,Mehta,Pune
7,Divya,Nair,Kochi


### Q3. List all unique categories in the products table.

In [7]:
show("SELECT DISTINCT category FROM products")

,category
0,Electronics
1,Clothing
2,Home


### Q4. Primary Key of each table — Explanation

| Table | Primary Key |
|-------|-------------|
| customers | customer_id |
| products | product_id |
| orders | order_id |
| order_items | item_id |

**Why must a Primary Key be UNIQUE and NOT NULL?**

A Primary Key uniquely identifies each row in a table. It must be UNIQUE because if two rows shared the same primary key, the database would not be able to tell them apart — leading to incorrect data retrieval. It must be NOT NULL because a NULL means "unknown", and you cannot uniquely identify a row with an unknown value.


### Q5. Constraints on the email column in customers table

The `email` column has two constraints:
1. **UNIQUE** — no two customers can have the same email address.
2. **NOT NULL** — every customer must provide an email.

If a duplicate email is inserted, the database raises: `UNIQUE constraint failed: customers.email` and rejects the insert. Demonstrated below:


In [8]:
# testing duplicate email insert
try:
    cur.execute("INSERT INTO customers VALUES (109,'Test','User','aarav.s@email.com','Mumbai','Maharashtra','2024-09-01',0)")
    conn.commit()
except Exception as e:
    print("Error:", e)


Error: UNIQUE constraint failed: customers.email


### Q6. Inserting a product with unit_price = -50

**INSERT statement attempted:**
```sql
INSERT INTO products VALUES (209, 'Broken Item', 'Electronics', 'TestBrand', -50.00, 100);
```
**Result:** The database raises `CHECK constraint failed: products`

**Constraint that prevents it:** `CHECK (unit_price > 0)` — negative prices are not valid for any product.


In [9]:
# testing CHECK constraint violation with negative price
try:
    cur.execute("INSERT INTO products VALUES (209,'Test Item','Electronics','Brand',-50.00,100)")
    conn.commit()
except Exception as e:
    print("Error:", e)
    print("Reason: CHECK constraint (unit_price > 0) failed")


Error: CHECK constraint failed: unit_price > 0
Reason: CHECK constraint (unit_price > 0) failed


---
## Section B — Filtering & Optimization

### Q7. Retrieve all orders with status = 'Delivered'.

In [10]:
show("SELECT * FROM orders WHERE status = 'Delivered'")

,order_id,customer_id,order_date,status,total_amount
0,1001,101,2024-08-01,Delivered,4498.0
1,1002,102,2024-08-03,Delivered,799.0
2,1004,101,2024-08-10,Delivered,3499.0
3,1006,105,2024-08-15,Delivered,5898.0
4,1008,103,2024-08-20,Delivered,899.0
5,1010,108,2024-08-28,Delivered,1598.0


### Q8. Products in Electronics category with unit_price > 2000.

In [11]:
show("""
SELECT product_name, brand, unit_price
FROM products
WHERE category = 'Electronics' AND unit_price > 2000
ORDER BY unit_price DESC
""")

,product_name,brand,unit_price
0,Bluetooth Speaker,JBL,3499.0
1,Smart Watch,Noise,2999.0


### Q9. Customers who joined in 2024 and belong to Maharashtra.

In [12]:
# Note: YEAR() function is not supported in SQLite, so using date range comparison instead
show("""
SELECT customer_id, first_name, last_name, city, state, join_date
FROM customers
WHERE join_date >= '2024-01-01' AND join_date < '2025-01-01'
  AND state = 'Maharashtra'
""")

,customer_id,first_name,last_name,city,state,join_date
0,101,Aarav,Sharma,Mumbai,Maharashtra,2024-01-15
1,107,Karan,Mehta,Pune,Maharashtra,2024-07-22


### Q10. Orders between 2024-08-10 and 2024-08-25 that are NOT cancelled.

In [13]:
show("""
SELECT * FROM orders
WHERE order_date BETWEEN '2024-08-10' AND '2024-08-25'
  AND status != 'Cancelled'
ORDER BY order_date
""")

,order_id,customer_id,order_date,status,total_amount
0,1004,101,2024-08-10,Delivered,3499.0
1,1006,105,2024-08-15,Delivered,5898.0
2,1007,106,2024-08-18,Pending,1299.0
3,1008,103,2024-08-20,Delivered,899.0
4,1009,107,2024-08-25,Shipped,6098.0


### Q11. What does idx_orders_date index do?

`CREATE INDEX idx_orders_date ON orders(order_date);`

This creates a B-tree index on the `order_date` column. Without an index, filtering by date requires a full table scan — the database reads every row. With the index, it jumps directly to matching rows, which is significantly faster on large datasets.

**Query that benefits from this index:**
```sql
SELECT * FROM orders
WHERE order_date BETWEEN '2024-08-01' AND '2024-08-15';
```


### Q12. Does YEAR(join_date) = 2024 use the index?

**No, it does not.** When a function like `YEAR()` is applied on a column, the database has to evaluate the function for every row first, which prevents index usage. This is called a non-SARGable predicate.

**Index-friendly (SARGable) rewrite:**
```sql
-- Non-SARGable (index NOT used):
SELECT * FROM customers WHERE YEAR(join_date) = 2024;

-- SARGable (index CAN be used):
SELECT * FROM customers
WHERE join_date >= '2024-01-01' AND join_date < '2025-01-01';
```


---
## Section C — Aggregation

### Q13. Total number of orders.

In [14]:
show("SELECT COUNT(*) AS total_orders FROM orders")

,total_orders
0,10


### Q14. Total revenue from all Delivered orders.

In [15]:
show("""
SELECT SUM(total_amount) AS total_revenue
FROM orders
WHERE status = 'Delivered'
""")

,total_revenue
0,17191.0


### Q15. Average unit_price per category.

In [16]:
show("""
SELECT category, ROUND(AVG(unit_price), 2) AS avg_price
FROM products
GROUP BY category
ORDER BY avg_price DESC
""")

,category,avg_price
0,Clothing,2699.0
1,Electronics,2224.0
2,Home,949.0


### Q16. Order count and total revenue by status, sorted by revenue.

In [17]:
show("""
SELECT status,
       COUNT(*) AS order_count,
       SUM(total_amount) AS total_revenue
FROM orders
GROUP BY status
ORDER BY total_revenue DESC
""")

,status,order_count,total_revenue
0,Delivered,6,17191.0
1,Shipped,2,13596.0
2,Cancelled,1,2999.0
3,Pending,1,1299.0


### Q17. Most expensive and cheapest product per category.

In [18]:
show("""
SELECT category,
       MAX(unit_price) AS most_expensive,
       MIN(unit_price) AS cheapest
FROM products
GROUP BY category
""")

,category,most_expensive,cheapest
0,Clothing,4599.0,799.0
1,Electronics,3499.0,899.0
2,Home,1299.0,599.0


### Q18. Categories with average unit_price greater than 2000 (HAVING clause).

In [19]:
show("""
SELECT category, ROUND(AVG(unit_price), 2) AS avg_price
FROM products
GROUP BY category
HAVING AVG(unit_price) > 2000
""")

,category,avg_price
0,Clothing,2699.0
1,Electronics,2224.0


---
## Section D — Joins & Relationships

### Q19. INNER JOIN — orders with customer name.

In [20]:
show("""
SELECT o.order_id, o.order_date,
       c.first_name, c.last_name,
       o.total_amount
FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id
ORDER BY o.order_date
""")

,order_id,order_date,first_name,last_name,total_amount
0,1001,2024-08-01,Aarav,Sharma,4498.0
1,1002,2024-08-03,Priya,Patel,799.0
2,1003,2024-08-05,Rohan,Gupta,7498.0
3,1004,2024-08-10,Aarav,Sharma,3499.0
4,1005,2024-08-12,Sneha,Reddy,2999.0
5,1006,2024-08-15,Vikram,Singh,5898.0
6,1007,2024-08-18,Ananya,Iyer,1299.0
7,1008,2024-08-20,Rohan,Gupta,899.0
8,1009,2024-08-25,Karan,Mehta,6098.0
9,1010,2024-08-28,Divya,Nair,1598.0


### Q20. LEFT JOIN — all customers and their orders (NULL if no order).

In [21]:
show("""
SELECT c.customer_id, c.first_name, c.last_name,
       o.order_id, o.order_date, o.total_amount
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
ORDER BY c.customer_id
""")

,customer_id,first_name,last_name,order_id,order_date,total_amount
0,101,Aarav,Sharma,1001,2024-08-01,4498.0
1,101,Aarav,Sharma,1004,2024-08-10,3499.0
2,102,Priya,Patel,1002,2024-08-03,799.0
3,103,Rohan,Gupta,1003,2024-08-05,7498.0
4,103,Rohan,Gupta,1008,2024-08-20,899.0
5,104,Sneha,Reddy,1005,2024-08-12,2999.0
6,105,Vikram,Singh,1006,2024-08-15,5898.0
7,106,Ananya,Iyer,1007,2024-08-18,1299.0
8,107,Karan,Mehta,1009,2024-08-25,6098.0
9,108,Divya,Nair,1010,2024-08-28,1598.0


### Q21. Three-table JOIN — order items with product details.

In [22]:
show("""
SELECT oi.order_id, p.product_name,
       oi.quantity, oi.unit_price, oi.discount_pct
FROM order_items oi
INNER JOIN orders o   ON oi.order_id   = o.order_id
INNER JOIN products p ON oi.product_id = p.product_id
ORDER BY oi.order_id
""")

,order_id,product_name,quantity,unit_price,discount_pct
0,1001,Wireless Earbuds,2,1499.0,0.0
1,1001,Laptop Stand,1,899.0,10.0
2,1002,Cotton T-Shirt,1,799.0,0.0
3,1003,Smart Watch,1,2999.0,0.0
4,1003,Running Shoes,1,4599.0,5.0
5,1004,Bluetooth Speaker,1,3499.0,0.0
6,1005,Smart Watch,1,2999.0,0.0
7,1006,Wireless Earbuds,1,1499.0,10.0
8,1006,Running Shoes,1,4599.0,5.0
9,1007,Bedsheet Set,1,1299.0,0.0


### Q22. LEFT JOIN vs RIGHT JOIN — explanation with example

| JOIN Type | Description |
|-----------|-------------|
| LEFT JOIN | All rows from left table + matching rows from right (NULL if no match) |
| RIGHT JOIN | All rows from right table + matching rows from left (NULL if no match) |
| FULL OUTER JOIN | All rows from both tables regardless of match |

**Example:**
```sql
-- LEFT JOIN: all customers, even those with no orders
SELECT c.first_name, o.order_id
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id;

-- RIGHT JOIN: all orders, even if customer record is missing
SELECT c.first_name, o.order_id
FROM customers c
RIGHT JOIN orders o ON c.customer_id = o.customer_id;
```

**When to use FULL OUTER JOIN:** When you need all records from both tables — for example, finding customers who never ordered AND orders that have no matching customer (useful in data audits).

*Note: SQLite does not support RIGHT JOIN or FULL OUTER JOIN natively.*


### Q23. Foreign Key relationships and referential integrity

| Table | Column | References |
|-------|--------|------------|
| orders | customer_id | customers(customer_id) |
| order_items | order_id | orders(order_id) |
| order_items | product_id | products(product_id) |

**What happens if customer_id = 999 is inserted (does not exist)?**
The database raises `FOREIGN KEY constraint failed` and rejects the insert. This protects referential integrity — an order must always belong to a valid customer.


In [23]:
cur.execute("PRAGMA foreign_keys = ON")
try:
    cur.execute("INSERT INTO orders VALUES (1099, 999, '2024-09-01', 'Pending', 500.00)")
    conn.commit()
except Exception as e:
    print("Error:", e)


---
## Section E — Advanced Concepts

### Q24. CASE — classify products into price tiers.

In [24]:
show("""
SELECT product_name, unit_price,
       CASE
           WHEN unit_price < 1000 THEN 'Budget'
           WHEN unit_price BETWEEN 1000 AND 3000 THEN 'Mid-Range'
           WHEN unit_price > 3000 THEN 'Premium'
       END AS price_tier
FROM products
ORDER BY unit_price
""")

,product_name,unit_price,price_tier
0,Cushion Covers (Set),599.0,Budget
1,Cotton T-Shirt,799.0,Budget
2,Laptop Stand,899.0,Budget
3,Bedsheet Set,1299.0,Mid-Range
4,Wireless Earbuds,1499.0,Mid-Range
5,Smart Watch,2999.0,Mid-Range
6,Bluetooth Speaker,3499.0,Premium
7,Running Shoes,4599.0,Premium


### Q25. CASE inside aggregate — Delivered vs Not Delivered count.

In [25]:
show("""
SELECT
    SUM(CASE WHEN status = 'Delivered' THEN 1 ELSE 0 END) AS delivered,
    SUM(CASE WHEN status != 'Delivered' THEN 1 ELSE 0 END) AS not_delivered
FROM orders
""")

,delivered,not_delivered
0,6,5


### Q26. ACID Properties

**A — Atomicity**
A transaction is all-or-nothing. If any step fails, the entire transaction is rolled back.
*Example:* In a bank transfer, if money is debited from Account A but the credit to Account B fails, the debit must also be undone. Either both happen or neither does.

**C — Consistency**
A transaction brings the database from one valid state to another. All constraints must hold after the transaction.
*Example:* After a transfer, the total money across both accounts must remain the same — no money is created or lost.

**I — Isolation**
Concurrent transactions do not interfere with each other. Each transaction sees a consistent view of the data.
*Example:* If two users buy the last item in stock simultaneously, isolation ensures only one succeeds — neither sees the other's in-progress changes.

**D — Durability**
Once a transaction is committed, the data is permanently saved even if the system crashes immediately after.
*Example:* After a payment is confirmed, a power failure will not revert the transaction — it is written to disk.


### Q27. Transaction — insert order, add items, update stock, rollback on failure.

In [26]:
cur.execute("PRAGMA foreign_keys = ON")

try:
    conn.execute("BEGIN")

    # insert new order
    conn.execute("INSERT INTO orders VALUES (1011, 102, date('now'), 'Pending', 1898.00)")

    # insert order items
    conn.execute("INSERT INTO order_items VALUES (5016, 1011, 206, 1, 1299.00, 0)")
    conn.execute("INSERT INTO order_items VALUES (5017, 1011, 208, 1, 599.00, 0)")

    # reduce stock for purchased products
    conn.execute("UPDATE products SET stock_qty = stock_qty - 1 WHERE product_id = 206")
    conn.execute("UPDATE products SET stock_qty = stock_qty - 1 WHERE product_id = 208")

    conn.execute("COMMIT")
    print("Transaction committed successfully.")

except Exception as e:
    conn.execute("ROLLBACK")
    print("Transaction rolled back due to error:", e)


Transaction committed successfully.


In [27]:
# verify results
print("New order inserted:")
print(show("SELECT * FROM orders WHERE order_id = 1011").to_string(index=False))
print("\nUpdated stock quantities:")
print(show("SELECT product_id, product_name, stock_qty FROM products WHERE product_id IN (206, 208)").to_string(index=False))


New order inserted:
 order_id  customer_id order_date  status  total_amount
     1011          102 2026-06-28 Pending        1898.0

Updated stock quantities:
 product_id         product_name  stock_qty
        206         Bedsheet Set        299
        208 Cushion Covers (Set)        399


---
## Brief Insights

1. **Customer base:** 8 customers across 7 Indian states; 4 are premium members.
2. **Product catalog:** 8 products across 3 categories. Electronics has the highest average price (~₹2,224), Home has the lowest (~₹949).
3. **Order performance:** 6 out of 10 orders are Delivered. Total delivered revenue = ₹17,191. Only 1 order was Cancelled (worth ₹2,999).
4. **Top customer:** Aarav Sharma (customer 101) placed 2 orders — the most among all customers.
5. **Popular products:** Wireless Earbuds and Running Shoes appear most frequently in order items.
6. **Price tiers:** 2 products are Budget (< ₹1000), 3 are Mid-Range (₹1000–3000), and 3 are Premium (> ₹3000).
